# Text-to-SQL with AgentCore and Postgres MCP

This notebook demonstrates querying a PostgreSQL analytics database through two Postgres MCP servers running on AWS AgentCore.

**Infrastructure used:**
- **AgentCore runtime** — stack `demo-postgres-mcp-agent` (deployed via `deploy_agentcore_agent_cf.sh`)
- **Views MCP server** — stack `demo-postgres-mcp-views` — tenant mode, scoped to allowed views/tenants (deployed via `deploy_views_mcp_cf.sh`)
- **Admin MCP server** — stack `demo-postgres-mcp-admin` — admin mode, full database access, no tenant restrictions (deployed via `deploy_admin_mcp_cf.sh`)

## Dependencies

In [1]:
%pip install --quiet boto3
print("✅ Dependencies installed")

Note: you may need to restart the kernel to use updated packages.
✅ Dependencies installed


## Imports

In [2]:
import boto3
import json
import uuid
import time
from typing import Dict, Any

print("✅ Imports loaded")

✅ Imports loaded


## Configuration

Pulls the AgentCore runtime ARN from the `demo-postgres-mcp-agent` stack and both MCP endpoints + API keys from their respective stacks automatically.

In [3]:
AWS_PROFILE = "tech-42"
AWS_REGION  = "us-east-1"

AGENTCORE_STACK_NAME  = "demo-postgres-mcp-agent"
VIEWS_MCP_STACK_NAME  = "demo-postgres-mcp-views"
ADMIN_MCP_STACK_NAME  = "demo-postgres-mcp-admin"

session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
cf      = session.client("cloudformation", region_name=AWS_REGION)
sm      = session.client("secretsmanager", region_name=AWS_REGION)

def _stack_output(stack_name: str, key: str) -> str:
    outputs = cf.describe_stacks(StackName=stack_name)["Stacks"][0]["Outputs"]
    return next(o["OutputValue"] for o in outputs if o["OutputKey"] == key)

# AgentCore runtime
AGENTCORE_RUNTIME_ARN = _stack_output(AGENTCORE_STACK_NAME, "AgentRuntimeArn")
AGENTCORE_MEMORY_ID   = _stack_output(AGENTCORE_STACK_NAME, "MemoryId")

# Views MCP (tenant mode)
VIEWS_MCP_URL     = _stack_output(VIEWS_MCP_STACK_NAME, "McpEndpointUrl")
VIEWS_MCP_API_KEY = sm.get_secret_value(
    SecretId=_stack_output(VIEWS_MCP_STACK_NAME, "ApiKeySecretArn")
)["SecretString"]

# Admin MCP (unrestricted)
ADMIN_MCP_URL     = _stack_output(ADMIN_MCP_STACK_NAME, "McpEndpointUrl")
ADMIN_MCP_API_KEY = sm.get_secret_value(
    SecretId=_stack_output(ADMIN_MCP_STACK_NAME, "ApiKeySecretArn")
)["SecretString"]

DEFAULT_MODEL_CONFIG = {
    "model_id": "us.anthropic.claude-haiku-4-5-20251001-v1:0",
    "region_name": AWS_REGION,
    "temperature": 0,
    "max_tokens": 4096,
}

print(f"✅ AgentCore runtime : {AGENTCORE_RUNTIME_ARN}")
print(f"✅ AgentCore memory  : {AGENTCORE_MEMORY_ID}")
print(f"✅ Views MCP         : {VIEWS_MCP_URL}")
print(f"✅ Admin MCP         : {ADMIN_MCP_URL}")
print(f"✅ API keys loaded from Secrets Manager")

✅ AgentCore runtime : arn:aws:bedrock-agentcore:us-east-1:123482425013:runtime/demo_mcp_agent_dev_d4c70040-CRgywn9yPr
✅ AgentCore memory  : demo_mcp_agent_dev_AgentCoreMemory_d4c70040-uT376fAh7j
✅ Views MCP         : http://demo-p-LoadB-56cHWvy3O96r-1217728724.us-east-1.elb.amazonaws.com/mcp
✅ Admin MCP         : http://demo-p-LoadB-BKk4l9VdgKOL-680689716.us-east-1.elb.amazonaws.com/mcp
✅ API keys loaded from Secrets Manager


## Helper Functions

In [4]:
def make_session_id() -> str:
    return f"session_{uuid.uuid4().hex}_{int(time.time())}"


def views_mcp_config(tenant_id: str) -> Dict:
    """MCP config for the Views server — tenant-scoped, read-only views."""
    return {
        "postgres": {
            "transport": "streamable_http",
            "url": VIEWS_MCP_URL,
            "headers": {
                "x-api-key": VIEWS_MCP_API_KEY,
                "x-tenant-id": tenant_id,
            },
        }
    }


def admin_mcp_config() -> Dict:
    """MCP config for the Admin server — full database access, no tenant restrictions."""
    return {
        "postgres": {
            "transport": "streamable_http",
            "url": ADMIN_MCP_URL,
            "headers": {
                "x-api-key": ADMIN_MCP_API_KEY,
            },
        }
    }


def invoke_text_to_sql(
    client,
    query: str,
    mcp_config: Dict,
    session_id: str = None,
    tenant_id: str = None,
    actor_id: str = "user-1",
    user_id: str = "demo",
    model_config: Dict = None,
    memory_id: str = None,
) -> Dict[str, Any]:
    """Invoke the AgentCore runtime with a text-to-SQL query via the Postgres MCP tool."""
    session_id   = session_id or make_session_id()
    model_config = model_config or DEFAULT_MODEL_CONFIG

    payload = {
        "query":        query,
        "model_config": model_config,
        "mcp_config":   mcp_config,
        "session_id":   session_id,
        "actor_id":     actor_id,
        "user_id":      user_id,
        "memory_id":    memory_id or AGENTCORE_MEMORY_ID,
    }
    if tenant_id:
        payload["tenant_id"] = tenant_id

    label = f"[{tenant_id}] " if tenant_id else "[admin] "
    print(f"📤 {label}{query}")
    print(f"   session: {session_id}")

    try:
        t0   = time.time()
        ttft = None

        response = client.invoke_agent_runtime(
            agentRuntimeArn=AGENTCORE_RUNTIME_ARN,
            runtimeSessionId=session_id,
            payload=json.dumps(payload).encode("utf-8"),
        )

        content = []
        print("\n🤖 Response:")
        print("-" * 80)
        for raw in response["response"].iter_lines(chunk_size=1):
            if not raw:
                continue
            if ttft is None:
                ttft = time.time() - t0  # first non-empty chunk

            line = raw.decode("utf-8")
            if line.startswith("data: "):
                line = line[6:]
                try:
                    line = json.loads(line)
                except Exception:
                    pass
                if isinstance(line, str) and line.startswith("data: "):
                    line = line[6:].rstrip("\n")
            print(line, flush=True)
            content.append(line)
        print("-" * 80)

        final = ""
        for line in content:
            try:
                d = json.loads(line) if isinstance(line, str) else line
                if d.get("type") == "final" and "response" in d:
                    final = d["response"]
                    break
            except Exception:
                pass
        if not final:
            for line in content:
                try:
                    d = json.loads(line) if isinstance(line, str) else line
                    if d.get("type") == "content_delta":
                        final += d.get("content", "")
                except Exception:
                    pass

        duration = time.time() - t0
        print(f"\n⏱  TTFT: {ttft:.2f}s  |  Total: {duration:.2f}s")
        print(f"\n📋 Answer:\n{final}\n")
        return {
            "success": True,
            "answer": final,
            "ttft": ttft,
            "duration": duration,
            "session_id": session_id,
        }

    except Exception as e:
        import traceback
        print(f"❌ Error: {e}")
        traceback.print_exc()
        return {"success": False, "answer": str(e), "ttft": None, "duration": 0, "session_id": session_id}


print("✅ Helper functions loaded")

✅ Helper functions loaded


## Initialize Client

In [5]:
client = session.client("bedrock-agentcore", region_name=AWS_REGION)
print(f"✅ bedrock-agentcore client ready (profile={AWS_PROFILE}, region={AWS_REGION})")

✅ bedrock-agentcore client ready (profile=tech-42, region=us-east-1)


---
## Views MCP Examples

The Views MCP runs in `tenant` mode — requests are scoped by `x-tenant-id` header and restricted to pre-approved views and tenant IDs.

### Example V1 — Tenant-Level KPI Query (tenant-a, expected to succeed)

In [6]:
result_v1 = invoke_text_to_sql(
    client=client,
    query="Use the Postgres MCP tool to get the order count and gross revenue KPI for me",
    mcp_config=views_mcp_config("tenant-a"),
    tenant_id="tenant-a",
    session_id=make_session_id(),
    actor_id="user-1",
    user_id="daan",
)

📤 [tenant-a] Use the Postgres MCP tool to get the order count and gross revenue KPI for me
   session: session_ee6a45d3f34c410a9fc252600a084def_1781452565

🤖 Response:
--------------------------------------------------------------------------------
{"type": "content_delta", "content": "I'll help", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTokens": 0}, "cycleCount": 1}
{"type": "content_delta", "content": " you get the order count and gross revenue", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTokens": 0}, "cycleCount": 1}
{"type": "content_delta", "content": " KPI.", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTokens": 0}, "cycleCount": 1}
{"type": "content_delta", "content": " First", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTok

### Example V2 — Non-Allowed Tenant Query (tenant-z, expected to fail)

`tenant-z` is not in the server's `TENANT_ALLOWED_TENANT_IDS` list — the MCP server should reject the request.

In [7]:
result_v2 = invoke_text_to_sql(
    client=client,
    query="Use the Postgres MCP tool to get the order count and gross revenue KPI for this tenant.",
    mcp_config=views_mcp_config("tenant-z"),
    tenant_id="tenant-z",
    session_id=make_session_id(),
    actor_id="user-1",
    user_id="daan",
)
print(f"⚠️  Expected failure — success={result_v2['success']}")

📤 [tenant-z] Use the Postgres MCP tool to get the order count and gross revenue KPI for this tenant.
   session: session_1d2d7988bb2f4fedb14d270296965f1c_1781452578

🤖 Response:
--------------------------------------------------------------------------------
{"type": "error", "message": "Failed to load tool <strands.tools.mcp.mcp_client.MCPClient object at 0xffff843d52e0>: Failed to start MCP client: the client initialization failed", "error_type": "ValueError"}
--------------------------------------------------------------------------------

⏱  TTFT: 1.28s  |  Total: 1.29s

📋 Answer:


⚠️  Expected failure — success=True


### Example V3 — List Accessible Tables (Views MCP)

Ask the Views MCP what tables and views it can see — should return only the pre-approved analytics views.

In [8]:
result_v3 = invoke_text_to_sql(
    client=client,
    query="Use the Postgres MCP tool to list all tables and views you have access to. Provide the exact view/tables names in your answer.",
    mcp_config=views_mcp_config("tenant-a"),
    tenant_id="tenant-a",
    session_id=make_session_id(),
    actor_id="user-1",
    user_id="daan",
)

📤 [tenant-a] Use the Postgres MCP tool to list all tables and views you have access to. Provide the exact view/tables names in your answer.
   session: session_98c2e2b0acc5459ca819646ca69af304_1781452591

🤖 Response:
--------------------------------------------------------------------------------
{"type": "content_delta", "content": "I'll list", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTokens": 0}, "cycleCount": 1}
{"type": "content_delta", "content": " all the", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTokens": 0}, "cycleCount": 1}
{"type": "content_delta", "content": " tables and views available to", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTokens": 0}, "cycleCount": 1}
{"type": "content_delta", "content": " you", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheRe

### Example V4 — Write Operations: INSERT and DELETE (expected to fail)

The Views MCP connects with a read-only database user — INSERT and DELETE should be rejected at the database level.

In [9]:
result_v4_insert = invoke_text_to_sql(
    client=client,
    query="Use the Postgres MCP tool to insert a test row into analytics_app.executive_kpis with all columns set to 0 or empty string.",
    mcp_config=views_mcp_config("tenant-a"),
    tenant_id="tenant-a",
    session_id=make_session_id(),
    actor_id="user-1",
    user_id="daan",
)
print(f"⚠️  INSERT — expected failure — success={result_v4_insert['success']}")

result_v4_delete = invoke_text_to_sql(
    client=client,
    query="Use the Postgres MCP tool to delete all rows from analytics_app.executive_kpis.",
    mcp_config=views_mcp_config("tenant-a"),
    tenant_id="tenant-a",
    session_id=make_session_id(),
    actor_id="user-1",
    user_id="daan",
)
print(f"⚠️  DELETE — expected failure — success={result_v4_delete['success']}")

📤 [tenant-a] Use the Postgres MCP tool to insert a test row into analytics_app.executive_kpis with all columns set to 0 or empty string.
   session: session_0e74b834c92944cfb6f88c1951f6cd42_1781452615

🤖 Response:
--------------------------------------------------------------------------------
{"type": "content_delta", "content": "I appreciate", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTokens": 0}, "cycleCount": 1}
{"type": "content_delta", "content": " your request, but I need to clar", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTokens": 0}, "cycleCount": 1}
{"type": "content_delta", "content": "ify the", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTokens": 0}, "cycleCount": 1}
{"type": "content_delta", "content": " capabilities", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 

---
## Admin MCP Examples

The Admin MCP runs in `admin` mode — no tenant restrictions, full database access.

### Example A1 — List All Accessible Tables (Admin MCP)

Should return every table and view in the database, across all schemas.

In [10]:
result_a1 = invoke_text_to_sql(
    client=client,
    query="Use the Postgres MCP tool to list all tables and views available in the database, grouped by schema. Provide the exact view/tables names in your answer.",
    mcp_config=admin_mcp_config(),
    session_id=make_session_id(),
    actor_id="user-1",
    user_id="daan",
)

📤 [admin] Use the Postgres MCP tool to list all tables and views available in the database, grouped by schema. Provide the exact view/tables names in your answer.
   session: session_48c505d3cae448f98712c434683a7ac9_1781452644

🤖 Response:
--------------------------------------------------------------------------------
{"type": "content_delta", "content": "I'll help", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTokens": 0}, "cycleCount": 1}
{"type": "content_delta", "content": " you list all tables and views in", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTokens": 0}, "cycleCount": 1}
{"type": "content_delta", "content": " the database,", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTokens": 0}, "cycleCount": 1}
{"type": "content_delta", "content": " groupe", "usage": {"inputTokens": 0, "outputTok

### Example A2 — Get Database Health Overview

Ask the Admin MCP to check the health of the database and surface any issues.

In [11]:
result_a2 = invoke_text_to_sql(
    client=client,
    query="Use the Postgres MCP tool to check the health of my database and identify any issues.",
    mcp_config=admin_mcp_config(),
    session_id=make_session_id(),
    actor_id="user-1",
    user_id="daan",
)

📤 [admin] Use the Postgres MCP tool to check the health of my database and identify any issues.
   session: session_a72635a457054e58b70e9966106e93fe_1781452675

🤖 Response:
--------------------------------------------------------------------------------
{"type": "content_delta", "content": "I'll analyze", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTokens": 0}, "cycleCount": 1}
{"type": "content_delta", "content": " the", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTokens": 0}, "cycleCount": 1}
{"type": "content_delta", "content": " health", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTokens": 0}, "cycleCount": 1}
{"type": "content_delta", "content": " of your database", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTokens": 0}, "cycleC

### Example A3 — Analyze Slow Queries

Ask the Admin MCP to surface the slowest queries and suggest how to speed them up.

In [12]:
result_a3 = invoke_text_to_sql(
    client=client,
    query="Use the Postgres MCP tool to find the slowest queries in my database. What are they, and how can I speed them up?",
    mcp_config=admin_mcp_config(),
    session_id=make_session_id(),
    actor_id="user-1",
    user_id="daan",
)

📤 [admin] Use the Postgres MCP tool to find the slowest queries in my database. What are they, and how can I speed them up?
   session: session_9bc3457b238f4c40ba4701b628755be4_1781452684

🤖 Response:
--------------------------------------------------------------------------------
{"type": "content_delta", "content": "I'll help", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTokens": 0}, "cycleCount": 1}
{"type": "content_delta", "content": " you find the slowest queries in your", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTokens": 0}, "cycleCount": 1}
{"type": "content_delta", "content": " database an", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTokens": 0}, "cycleCount": 1}
{"type": "content_delta", "content": "d provide", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheRe

### Example A4 — Get Recommendations On How To Speed Things Up

Ask the Admin MCP for general performance recommendations when the application feels slow.

In [13]:
result_a4 = invoke_text_to_sql(
    client=client,
    query="Use the Postgres MCP tool. My app is slow. How can I make it faster? Analyze the database and give me concrete recommendations.",
    mcp_config=admin_mcp_config(),
    session_id=make_session_id(),
    actor_id="user-1",
    user_id="daan",
)

📤 [admin] Use the Postgres MCP tool. My app is slow. How can I make it faster? Analyze the database and give me concrete recommendations.
   session: session_8cb47543090248d49bc52a9c2f9c4a13_1781452701

🤖 Response:
--------------------------------------------------------------------------------
{"type": "content_delta", "content": "I'll help", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTokens": 0}, "cycleCount": 1}
{"type": "content_delta", "content": " you analyze", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTokens": 0}, "cycleCount": 1}
{"type": "content_delta", "content": " your database and provide concrete recommendations to make", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTokens": 0}, "cycleCount": 1}
{"type": "content_delta", "content": " your app faster. Let me start by", "usage": {"in

### Example A5 — Generate Index Recommendations

Ask the Admin MCP to analyze the workload and suggest indexes that would improve query performance.

In [14]:
result_a5 = invoke_text_to_sql(
    client=client,
    query="Use the Postgres MCP tool to analyze my database workload and suggest indexes to improve performance.",
    mcp_config=admin_mcp_config(),
    session_id=make_session_id(),
    actor_id="user-1",
    user_id="daan",
)

📤 [admin] Use the Postgres MCP tool to analyze my database workload and suggest indexes to improve performance.
   session: session_9d07a1fb84ea4c1dafcf101bc7ef25c5_1781452725

🤖 Response:
--------------------------------------------------------------------------------
{"type": "content_delta", "content": "I'll analyze", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTokens": 0}, "cycleCount": 1}
{"type": "content_delta", "content": " your database workload an", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTokens": 0}, "cycleCount": 1}
{"type": "content_delta", "content": "d suggest indexes to improve performance.", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTokens": 0}, "cycleCount": 1}
{"type": "tool_use_start", "tool": "analyze_workload_indexes", "tool_use_id": "tooluse_7DTCvTTfaHLGR6x8nmILPX", "i

### Example A6 — Optimize a Specific Query

Ask the Admin MCP to optimize a query from the demo database.
The query joins `raw_app.orders` to `raw_app.customers` and filters by `order_ts`, mirroring
a common reporting pattern on this schema.

In [15]:
result_a6 = invoke_text_to_sql(
    client=client,
    query=(
        "Use the Postgres MCP tool to help me optimize this query:\n"
        "SELECT * FROM raw_app.orders "
        "JOIN raw_app.customers ON raw_app.orders.customer_id = raw_app.customers.id "
        "WHERE raw_app.orders.order_ts > '2024-01-01';"
    ),
    mcp_config=admin_mcp_config(),
    session_id=make_session_id(),
    actor_id="user-1",
    user_id="daan",
)

📤 [admin] Use the Postgres MCP tool to help me optimize this query:
SELECT * FROM raw_app.orders JOIN raw_app.customers ON raw_app.orders.customer_id = raw_app.customers.id WHERE raw_app.orders.order_ts > '2024-01-01';
   session: session_af2015d342b94d29a764b5533d7b2ae8_1781452739

🤖 Response:
--------------------------------------------------------------------------------
{"type": "content_delta", "content": "I'll help you optimize this", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTokens": 0}, "cycleCount": 1}
{"type": "content_delta", "content": " query.", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTokens": 0}, "cycleCount": 1}
{"type": "content_delta", "content": " Let me start", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTokens": 0}, "cycleCount": 1}
{"type": "content_delta", "content": " 

---
## Views vs Raw — Speed Comparison

The Views path should be significantly faster end-to-end: less schema for the model to reason over, simpler SQL to generate, and a lighter database query to execute.

### P1 — Views 

In [16]:
result_p1_views = invoke_text_to_sql(
    client=client,
    query="Use the Postgres MCP tool to get the order count and gross revenue KPI for tenant a",
    mcp_config=views_mcp_config("tenant-a"),
    tenant_id="tenant-a",
    session_id=make_session_id(),
    actor_id="user-1",
    user_id="daan",
)

📤 [tenant-a] Use the Postgres MCP tool to get the order count and gross revenue KPI for tenant a
   session: session_8964fae854734694b900d53cff00c987_1781452750

🤖 Response:
--------------------------------------------------------------------------------
{"type": "content_delta", "content": "I'll help", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTokens": 0}, "cycleCount": 1}
{"type": "content_delta", "content": " you get the order count and gross revenue", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTokens": 0}, "cycleCount": 1}
{"type": "content_delta", "content": " KPI for tenant a", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTokens": 0}, "cycleCount": 1}
{"type": "content_delta", "content": ".", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cache

### P2 — Same Query via Admin MCP (raw tables)

In [17]:
result_p2_admin = invoke_text_to_sql(
    client=client,
    query="Use the Postgres MCP tool to get the order count and gross revenue KPI for tenant a",
    mcp_config=admin_mcp_config(),
    session_id=make_session_id(),
    actor_id="user-1",
    user_id="daan",
)

📤 [admin] Use the Postgres MCP tool to get the order count and gross revenue KPI for tenant a
   session: session_0e52ea252c6c42259de0da4f079541c8_1781452761

🤖 Response:
--------------------------------------------------------------------------------
{"type": "content_delta", "content": "I'll help", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTokens": 0}, "cycleCount": 1}
{"type": "content_delta", "content": " you get the order count and gross revenue", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTokens": 0}, "cycleCount": 1}
{"type": "content_delta", "content": " KPI for tenant a", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWriteInputTokens": 0}, "cycleCount": 1}
{"type": "content_delta", "content": ".", "usage": {"inputTokens": 0, "outputTokens": 0, "totalTokens": 0, "cacheReadInputTokens": 0, "cacheWri

### Performance Comparison

In [18]:
views_ttft  = result_p1_views["ttft"]
views_total = result_p1_views["duration"]
admin_ttft  = result_p2_admin["ttft"]
admin_total = result_p2_admin["duration"]

print(f"{"":30} {"TTFT":>8} {"Total":>8}")
print("-" * 48)
print(f"{"P1 Views MCP (1 view)":30} {views_ttft:>7.2f}s {views_total:>7.2f}s")
print(f"{"P2 Admin MCP (7 raw tables)":30} {admin_ttft:>7.2f}s {admin_total:>7.2f}s")
print()
if admin_total and views_total:
    print(f"Admin was {admin_total / views_total:.1f}x slower end-to-end")

                                   TTFT    Total
------------------------------------------------
P1 Views MCP (1 view)             2.44s   11.39s
P2 Admin MCP (7 raw tables)       2.25s   16.38s

Admin was 1.4x slower end-to-end
